# Step 3 — Metrics by regime

**RSM434 Industry Project 3 · Option strategies through market regimes**

This notebook measures every strategy and the S&P 500, overall and in the month after each regime reading. It uses the panel saved in Step 2, so the regime labels are unchanged.

1. Load the panel and confirm the readings
2. Full-sample metrics
3. The four volatility regimes
4. Credit versus the 2s5s slope
5. Whether the result holds in both halves of the sample

The calculations live in `ip3.py`. Returns in the printed tables are in percent. Sharpe and beta are plain numbers.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

import ip3

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 24)
pd.set_option("display.max_rows", 80)

PERCENT = ["ann_return", "ann_vol", "max_drawdown", "var_95", "cvar_95", "worst_return", "total_return"]


def show(frame):
    out = frame.copy()
    for col in PERCENT:
        if col in out.columns:
            out[col] = (out[col] * 100).round(1)
    for col in ("sharpe", "beta", "sharpe_first", "sharpe_second", "beta_first", "beta_second"):
        if col in out.columns:
            out[col] = out[col].round(2)
    display(out)


daily, readings, paired = ip3.load_processed()
month_ends = ip3.month_end_calendar(daily)
check = ip3.month_end_reading_check(daily, month_ends, readings)
print(f"{len(paired)} return months, {paired.index.min():%b %Y} to {paired.index.max():%b %Y}.")
print("Month-end readings match a fresh calculation." if check.empty else check.head())


## Definitions

For each series, over the months in the cell:

- **Annualized return** is geometric: the product of `1 + monthly return`, raised to `12/n`, minus 1.
- **Annualized volatility** is the sample standard deviation of monthly returns, times `sqrt(12)`.
- **Sharpe** is the mean excess return over that month's T-bill return, divided by the standard deviation of the total return, times `sqrt(12)`. The T-bill return was known when the month began.
- **Max drawdown** compounds those months in calendar order, starting from 1. Regime months are not always consecutive, so this is the conditional track record. It is shown here as a negative number.
- **95% VaR** is minus the 5th percentile of monthly returns, and **95% CVaR** is minus the average of the months at or below it. Both are positive losses. A cell with fewer than 20 months is flagged, because that percentile is then one or two months.
- **Beta** is the slope on the S&P 500 total return over the same months. Buy-writes and the collar hold the equity. PUT, CNDR and BFLY hold T-bills.

Every return month is already paired with the previous month-end, so none of these averages uses a label from inside the month.


In [ ]:
metrics_all = ip3.performance_table(paired)
metrics_vol = ip3.performance_table(paired, "vol_regime")
metrics_credit = ip3.performance_table(paired, "regime_credit")
metrics_slope = ip3.performance_table(paired, "regime_slope")

print("Full sample")
show(metrics_all.drop(columns="group"))


## The four volatility regimes

The counts below are the number of return months that followed each reading. Low/Falling is the large calm bucket. High/Falling, the recovery bucket, is the thin one.


In [ ]:
def pivot(metric, frame=metrics_vol, columns=ip3.VOL_REGIMES):
    table = ip3.pivot_metric(frame, metric, columns)
    if metric not in ("sharpe", "beta", "n"):
        table = (table * 100).round(1)
    elif metric != "n":
        table = table.round(2)
    return table


for metric, title in [
    ("n", "Months"),
    ("ann_return", "Annualized return (%)"),
    ("ann_vol", "Annualized volatility (%)"),
    ("sharpe", "Sharpe"),
    ("max_drawdown", "Max drawdown, stitched (%)"),
    ("cvar_95", "95% CVaR, loss (%)"),
    ("beta", "Beta to the S&P 500"),
]:
    print(title)
    display(pivot(metric))

sharpe = ip3.pivot_metric(metrics_vol, "sharpe", ip3.VOL_REGIMES)
cvar = ip3.pivot_metric(metrics_vol, "cvar_95", ip3.VOL_REGIMES) * 100


def heatmap(table, title, path, cmap, vmin, vmax, fmt):
    fig, ax = plt.subplots(figsize=(8.4, 4.8))
    values = table.to_numpy(dtype=float)
    image = ax.imshow(values, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_xticks(range(table.shape[1]), table.columns, rotation=15, ha="right")
    ax.set_yticks(range(table.shape[0]), table.index)
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            ax.text(j, i, format(values[i, j], fmt), ha="center", va="center", fontsize=8)
    ax.set_title(title)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    ip3.OUTPUT.mkdir(parents=True, exist_ok=True)
    fig.savefig(ip3.OUTPUT / path, dpi=150)
    plt.show()


heatmap(sharpe, "Sharpe in the month after each volatility regime", "sharpe_by_regime.png", "RdYlGn", -1.0, 1.6, ".2f")
heatmap(cvar, "95% CVaR (loss, %) in the month after each volatility regime", "cvar_by_regime.png", "YlOrRd", 3, 14, ".1f")


The stress bucket is High/Rising, and its average equity return is still positive. The drawdown and CVaR columns are where protection separates from the buy-writes and the put-write. Beta moves with that result: in High/Rising, PPUT and CLL fall to about 0.56, while the buy-writes stay near 0.7 to 0.85. CNDR and BFLY have betas near zero in every bucket, so a loss there is the option book, not hidden equity.

BFLY's full-sample stitched drawdown is deeper than the S&P's, on a beta near zero. The loss is accumulated across ordinary months. In the stress bucket its CVaR is smaller than the S&P's, and its Sharpe is negative.


## Credit, and the slope as the comparison

The brief asks for one added signal, and for a test of whether it sharpens the picture. Both signals were built in Step 2. The tables here are the test. A cell under 12 months is shown and then set aside: Low/Rising with widening credit has 5 months, and High/Falling with tightening credit has 12, with an S&P return that is too large to treat as a forecast.


In [ ]:
credit_cols = ip3.REGIME_ORDER["regime_credit"]
slope_cols = ip3.REGIME_ORDER["regime_slope"]
print("Annualized return (%) by volatility regime and credit")
display(pivot("ann_return", metrics_credit, credit_cols))
print("Sharpe by volatility regime and credit")
display(pivot("sharpe", metrics_credit, credit_cols))
print("Annualized return (%) by volatility regime and 2s5s slope")
display(pivot("ann_return", metrics_slope, slope_cols))
print("Sharpe by volatility regime and 2s5s slope")
display(pivot("sharpe", metrics_slope, slope_cols))

halves = ip3.sample_halves(paired.index)
calm_wide = paired[(paired.vol_regime == "Low/Falling") & (paired.credit_signal == "Widening")]
print("Low/Falling and widening credit, S&P average monthly return by half of the sample")
for label in ("First half", "Second half"):
    mask = halves.reindex(calm_wide.index) == label
    block = calm_wide.loc[mask, "SPX_TR"]
    print(f"  {label}: {len(block)} months, mean {block.mean() * 100:.2f}%")


Credit changes the calm bucket and the stress bucket. Inside Low/Falling, subsequent S&P returns are strongly positive when credit is tightening and negative when it is widening. That widening cell is the reading in front of September 2008 and January 2022. Split the same 22 months in half and the gap shrinks: the first half of the sample carries most of the loss, and the second half is about flat. It is a reason to cut equity-collateral income, with low confidence.

Inside High/Rising, widening credit is where the put-write's Sharpe falls to about zero and PPUT's rises. Tightening credit puts the put-write back in line with the S&P. Both of those cells have more than 12 months.

The slope does not separate High/Rising equity returns. It does separate Low/Falling. Credit is the added signal because the strategy ranking moves in stress, and the slope leaves that ranking alone.


## Both halves of the sample

The sample is split into two equal sets of months, the same split as in Step 2. Return, Sharpe and beta are recomputed inside each volatility regime. A sign flip means the Sharpe is positive in one half and negative in the other. Rank 1 is the highest Sharpe among the eight strategies and the S&P in that half. Ranks move easily in a bucket of 30 months, so a sign flip matters more than a one-place rank change.


In [ ]:
stability = ip3.stability_table(paired)
cols = [
    "group", "asset", "n_first", "n_second", "ann_return_first", "ann_return_second",
    "sharpe_first", "sharpe_second", "beta_first", "beta_second",
    "excess_sign_flip", "sharpe_rank_flip", "half_thin",
]
print("Excess-return sign flips")
show(stability.loc[stability.excess_sign_flip, cols])
moved = stability.loc[stability.sharpe_rank_flip & ~stability.half_thin].copy()
moved["rank_change"] = (moved.sharpe_rank_first - moved.sharpe_rank_second).abs()
print("Sharpe-rank changes of 3 or more, with at least 6 months in each half")
show(moved.loc[moved.rank_change >= 3, cols + ["sharpe_rank_first", "sharpe_rank_second", "rank_change"]])


Three sign flips show up. BFLY in Low/Falling is positive in the first half and negative in the second. CNDR and BFLY in High/Falling do the same. High/Rising has no sign flip: PPUT stays positive in both halves, and CNDR and BFLY stay negative. That is the regime result to lean on. High/Falling has only 13 and 18 months in the two halves, and the ranks reshuffle, so a single Sharpe from that bucket is a weak reason to prefer one equity book over another.

Step 4 uses these tables for the four stress windows, the hidden Markov comparison, the monitor and the deck notes.
